In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,FloatType, DoubleType
from pyspark.sql import functions as F

catalog_name = 'ecommerce'
df_silver = spark.table(f"{catalog_name}.bronze.brz_order_items")
df_silver.printSchema()

# Remove duplicates
df_silver = df_silver.dropDuplicates(['order_id', 'item_seq'])

# Clean quantity
df_silver = df_silver.withColumn(
    'quantity',
    F.regexp_replace(F.col("quantity"), "Two", '2').cast('int')
)

# Clean unit_price
df_silver = df_silver.withColumn(
    'unit_price',
    F.regexp_replace(F.col('unit_price'), r"[^0-9.]", "").cast('double')
)

# Clean discount_pct
df_silver = df_silver.withColumn(
    'discount_pct',
    F.regexp_replace(F.col('discount_pct'), r"[^0-9.]", "").cast('double')
)

# Clean coupon_code
df_silver = df_silver.withColumn(
    'coupon_code',
    F.lower(F.trim(F.col('coupon_code')))
)

# Standardize channel
df_silver = df_silver.withColumn(
    'channel',
    F.when(F.col('channel') == 'web', 'Website')
    .when(F.col('channel') == 'app', 'Mobile')
    .otherwise(F.col('channel'))
)

# Convert date format (DD-MM-YYYY to standard)
df_silver = df_silver.withColumn(
    "dt",
    F.to_date(F.col("dt"), "yyyy-MM-dd")
)

# Convert timestamp (DD-MM-YYYY HH:mm to standard)
df_silver = df_silver.withColumn(
    "order_ts",
    F.to_timestamp(F.col("order_ts"), "yyyy-MM-dd HH:mm:ss")
)

# Convert item_seq to integer
df_silver = df_silver.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)

# Clean tax_amount
df_silver = df_silver.withColumn(
    "tax_amount",
    F.regexp_replace(F.col("tax_amount"), r"[^0-9.]", "").cast("double")
)

# Add processing timestamp
df_silver = df_silver.withColumn(
    "processed_time",
    F.current_timestamp()
)

# Write to Silver layer
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")

display(df_silver)